# 🧠 Pipeline de Machine Learning e Otimização Financeira: Detecção de Fraude

**Autor:** Douglas Moura / DOCHMO Analytics  
**Dataset:** Credit Card Fraud Detection (Kaggle / ULB Machine Learning Group)  
**Foco:** Modelagem preditiva sob desbalanceamento severo, benchmark de técnicas de reamostragem, otimização de métricas com foco em **Recall** e **PR-AUC**, calibração de limiares de corte (*Threshold Tuning*) e **quantificação do retorno financeiro real (ROI)**.

---

## 🎯 Roteiro do Pipeline de Modelagem
1. **Configuração do Ambiente e Módulos Estruturados (`src/`)**
2. **Carregamento e Pré-processamento com Escalonamento Robusto (`RobustScaler`)**
3. **Divisão Estratificada (Train/Test Split 80/20)** — *Prevenção Absoluta de Data Leakage*
4. **Tratamento Condicional de Outliers Extremos no Treinamento**
5. **Benchmark Comparativo de Modelos e Técnicas de Balanceamento:**
   * *Baseline (Sem Balanceamento)*
   * *Under-sampling (Random Under Sampler)*
   * *Over-sampling (SMOTE)*
   * *Ponderação de Classes (Cost-Sensitive / Class Weight)*
   * *Modelos Baseados em Árvores e Ensembles (Random Forest & XGBoost)*
6. **Painel Visual 3x3 de Performance: Matrizes de Confusão, Curvas ROC e PR**
7. **Otimização de Hiperparâmetros do Modelo Campeão (`RandomizedSearchCV`) & Ressalva Analítica**
8. **Análise de Importância das Variáveis (*Feature Importances*)**
9. **Calibração de Limiar de Decisão (*Threshold Tuning*) e Simulação de Custo Financeiro**
10. **Diagnóstico de Ticket (TP vs. FN) e Análise de Sensibilidade de Custos Operacionais**
11. **Resumo Executivo e Diretrizes para Deploy em Produção**


In [1]:
# 1. Importações de Bibliotecas Essenciais
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 2. Configuração de Caminhos e Importação dos Módulos Modulares em src/
sys.path.append(os.path.abspath(".."))
from src.preprocessing import load_data, prepare_and_scale_data, split_data
from src.outlier_detector import remove_extreme_outliers
from src.modeling import get_model_candidates, evaluate_candidates
from src.metrics import (
    evaluate_model,
    plot_model_performance_dashboard,
    plot_confusion_matrix_plotly,
    plot_feature_importances
)
from src.business import (
    calculate_financial_impact,
    optimize_threshold,
    plot_threshold_curves_plotly,
    plot_tp_vs_fn_amounts,
    simulate_fp_cost_sensitivity
)

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier

print("[INFO] Todos os módulos e dependências carregados com sucesso.")

[INFO] Todos os módulos e dependências carregados com sucesso.


## 1. Carregamento e Pré-processamento dos Dados

Como identificado no EDA, as variáveis `Time` e `Amount` possuem ordens de grandeza numéricas e desvios muito superiores aos das componentes principais `V1-V28`.  
Utilizamos o `RobustScaler` (que subtrai a mediana e divide pelo intervalo interquartil IQR), tornando a escala resistente a transações com valores atípicos ou extremos.


In [2]:
# 1. Carregamento da Base Bruta
raw_df = load_data("../data/creditcard.csv")

# 2. Preservação dos Valores Originais de 'Amount' para a Simulação Financeira
original_amounts = raw_df["Amount"].copy()

# 3. Aplicação do RobustScaler em Time e Amount
df_scaled = prepare_and_scale_data(raw_df)

print("--- Amostra das Features Processadas com RobustScaler ---")
display(df_scaled.head())

[INFO] Carregando dados de: ../data/creditcard.csv


[INFO] Dataset carregado com sucesso: 284,807 linhas e 31 colunas.
--- Amostra das Features Processadas com RobustScaler ---


,scaled_time,scaled_amount,V1,V2,V3,V4,V5,V6,V7,V8,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Class
0,-0.994983,1.783274,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,...,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,0
1,-0.994983,-0.269825,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,...,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,0
2,-0.994972,4.983721,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,...,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,0
3,-0.994972,1.418291,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,...,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,0
4,-0.994960,0.670579,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,...,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,0


## 2. Divisão Estratificada (Train/Test Split) — Prevenção de Data Leakage

> ⚠️ **Regra Metodológica Crítica:** Nenhuma técnica de balanceamento (RUS, SMOTE) ou remoção de outliers pode ser aplicada antes da partição dos dados.  
> O conjunto de teste deve permanecer **rigorosamente intocado** para simular o ambiente real de produção.


In [3]:
# 1. Divisão Estratificada 80/20 (Mantendo a Proporção Exata de Fraudes)
X_train, X_test, y_train, y_test = split_data(
    df_scaled,
    target_col="Class",
    test_size=0.20,
    random_state=42
)

# 2. Isolamento dos Valores de Amount do Conjunto de Teste para Avaliação de ROI
test_indices = y_test.index
test_amounts = original_amounts.loc[test_indices].values

print(f"Total de fraudes no conjunto de teste: {y_test.sum():,} de {len(y_test):,} transações avaliadas.")

[INFO] Treino: 227,845 amostras | Fraudes: 394 (0.173%)
[INFO] Teste:  56,962 amostras | Fraudes: 98 (0.172%)
Total de fraudes no conjunto de teste: 98 de 56,962 transações avaliadas.


## 3. Tratamento Condicional de Outliers Extremos no Treino

No EDA, a auditoria global de IQR revelou que as fraudes habitam a cauda externa extrema de variáveis como `V14`, `V12` e `V10`, comprovando que um corte global ingênuo eliminaria até 84% da própria classe minoritária.  

Por essa razão, na etapa de modelagem preditiva alteramos o critério de forma tecnicamente rigorosa:
- O cálculo do IQR ($2{,}5 	imes 	ext{IQR}$) é aplicado **estritamente dentro da distribuição das fraudes no conjunto de treino**, e não na base inteira.
- Esse procedimento corta exclusivamente transações com distorções aberrantes que deformariam os hiperplanos do classificador, preservando 388 fraudes autênticas e representativas para o aprendizado supervisionado.


In [4]:
# 1. Execução do Filtro Condicional no Treino
X_train_clean, y_train_clean = remove_extreme_outliers(
    X_train,
    y_train,
    features=['V14', 'V12', 'V10'],
    factor=2.5
)

[INFO] Tratamento Condicional de Outliers (fator 2.5):
       - Linhas totais de treino preservadas: 227,839 de 227,845
       - Fraudes extremas removidas: 6 | Fraudes mantidas: 388 (de 394)


## 4. Benchmark de Estratégias de Modelagem e Reamostragem

Para compreender o comportamento do algoritmo sob desbalanceamento severo, treinamos e comparamos 6 abordagens no conjunto de teste intocado:
1. **Regressão Logística (Baseline):** Modelo linear clássico sem qualquer tratamento de desbalanceamento.
2. **Regressão Logística (Under-sampling):** Redução aleatória da classe majoritária com `RandomUnderSampler` em pipeline.
3. **Regressão Logística (SMOTE):** Síntese de novas amostras da classe minoritária via interpolação k-NN.
4. **Regressão Logística (Class-Weighted):** Penalização proporcional por erro na classe minoritária (`class_weight='balanced'`).
5. **Random Forest (Class-Weighted):** Comitê de 100 árvores com ponderação balanceada por subamostra (*Bagging*).
6. **XGBoost (Cost-Sensitive):** Algoritmo de alta performance (*Gradient Boosting*) ponderado com custo proporcional aos erros na classe 1.


In [5]:
# 1. Obtenção dos Pipelines Candidatos com Parâmetros Calibrados
candidates = get_model_candidates(scale_pos_weight=10.0, random_state=42)

# 2. Treinamento e Avaliação no Conjunto de Teste Intocado
results_df, test_probabilities, trained_models = evaluate_candidates(
    candidates,
    X_train_clean,
    y_train_clean,
    X_test,
    y_test
)

[TREINANDO] 1. Logistic Regression (Baseline)...


[TREINANDO] 2. Logistic Regression (Under-sampling)...
[TREINANDO] 3. Logistic Regression (SMOTE)...


[TREINANDO] 4. Logistic Regression (Class-Weighted)...


[TREINANDO] 5. Random Forest (Class-Weighted)...


[TREINANDO] 6. XGBoost (Cost-Sensitive)...


In [6]:
# 1. Preparação da Tabela Formatada no Padrão Dark Theme (#0B1320)
styled_benchmark = results_df.style.format({
    'Recall': '{:.2%}',
    'Precision': '{:.2%}',
    'F1-Score': '{:.2%}',
    'PR-AUC': '{:.4f}',
    'ROC-AUC': '{:.4f}',
    'TP': '{:,}',
    'FP': '{:,}',
    'FN': '{:,}',
    'TN': '{:,}'
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#1E293B'), ('color', '#E2E8F0'), ('font-family', 'Arial'), ('font-size', '13px'), ('text-align', 'center'), ('border', '1px solid #334155')]},
    {'selector': 'td', 'props': [('background-color', '#0B1320'), ('color', '#E2E8F0'), ('font-family', 'Arial'), ('font-size', '13px'), ('text-align', 'center'), ('border', '1px solid #1E293B')]},
    {'selector': 'caption', 'props': [('caption-side', 'top'), ('color', '#E2E8F0'), ('font-size', '16px'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding-bottom', '10px')]}
]).set_properties(**{
    'font-weight': 'bold',
    'color': '#E2E8F0'
}, subset=['PR-AUC', 'F1-Score']).set_caption("Tabela Comparativa de Performance: Modelos e Estratégias de Reamostragem")

display(styled_benchmark)

,Modelo,Recall,Precision,F1-Score,PR-AUC,ROC-AUC,TP,FP,FN,TN
0,6. XGBoost (Cost-Sensitive),85.71%,86.60%,86.15%,0.8681,0.9741,84,13,14,"56,851"
1,5. Random Forest (Class-Weighted),81.63%,79.21%,80.40%,0.8088,0.9731,80,21,18,"56,843"
2,1. Logistic Regression (Baseline),64.29%,82.89%,72.41%,0.7409,0.9573,63,13,35,"56,851"
3,3. Logistic Regression (SMOTE),91.84%,5.78%,10.88%,0.7233,0.9715,90,"1,467",8,"55,397"
4,4. Logistic Regression (Class-Weighted),91.84%,6.02%,11.29%,0.7189,0.9721,90,"1,406",8,"55,458"
5,2. Logistic Regression (Under-sampling),91.84%,3.81%,7.32%,0.6940,0.9760,90,"2,270",8,"54,594"


### 🔍 Diagnóstico Técnico do Benchmark

> **Insight:** Ao confrontarmos os resultados empíricos da tabela acima com as restrições reais de uma operação bancária:
> * **Baseline (Regressão Logística sem balanceamento):** Apresenta Precisão aceitável (**82,89%**), mas um Recall de apenas **64,29%**, permitindo que **35 das 98 fraudes** passem despercebidas, gerando prejuízo financeiro direto e passivos de chargeback.
> * **Under-sampling (RUS):** Eleva o Recall para **91,84%** (deixando passar apenas 8 fraudes), porém destrói a Precisão (**3,81%**), gerando **2.270 alarmes falsos**. Para cada fraude legítima capturada, mais de 25 clientes inocentes teriam seus cartões bloqueados indevidamente.
> * **SMOTE e Class-Weighted (Modelos Lineares):** Ambos alcançam o mesmo Recall de **91,84%**, mas ainda geram um volume massivo de alarmes falsos (**1.467** e **1.406 FPs**, respectivamente), demonstrando que a separação linear é incapaz de isolar com precisão as caudas sobrepostas no espaço vetorial.
> * **Modelos de Ensembles (Random Forest e XGBoost):** Apresentam dominância técnica categórica. O **Random Forest** alcança **81,63% de Recall** com **79,21% de Precisão** e **0,8088 de PR-AUC**. O **XGBoost (Cost-Sensitive)** atinge a melhor performance global, combinando **85,71% de Recall**, **86,60% de Precisão** e o maior **PR-AUC (0,8681)** com apenas **13 alarmes falsos** em 56.864 transações legítimas.


## 5. Painel Visual Comparativo: Matrizes de Confusão e Curvas de Performance

Inspirado na arquitetura do projeto Titanic, o dashboard interativo abaixo cruza os três arquétipos centrais de modelagem:
1. **Regressão Logística (Modelo Linear Baseline)**
2. **Random Forest (Modelo de Ensemble - Bagging)**
3. **XGBoost (Modelo de Ensemble - Boosting Campeão)**

Para cada modelo avaliamos em uma única linha a **Matriz de Confusão**, a **Curva ROC** e a **Curva Precision-Recall**.


In [7]:
# 1. Seleção dos Três Modelos Representativos
selected_models = {
    'Logistic Regression': trained_models['1. Logistic Regression (Baseline)'],
    'Random Forest': trained_models['5. Random Forest (Class-Weighted)'],
    'XGBoost': trained_models['6. XGBoost (Cost-Sensitive)']
}

# 2. Renderização do Dashboard 3x3 no Plotly Dark
fig_dashboard = plot_model_performance_dashboard(
    models=selected_models,
    X_val=X_test,
    y_val=y_test,
    height=1000,
    width=1200
)

fig_dashboard.show()

## 6. Otimização Fina de Hiperparâmetros do Modelo Campeão (XGBoost)

Utilizamos o `RandomizedSearchCV` aliado à validação cruzada estratificada (`StratifiedKFold(n_splits=3)`) para explorar hiperparâmetros críticos: taxa de aprendizado (`learning_rate`), profundidade máxima das árvores (`max_depth`), amostragem de linhas/colunas e ponderação de custo (`scale_pos_weight`), tendo como função-alvo a maximização do **PR-AUC** (`average_precision`).

> 📌 **Nota Metodológica:** Foi adotado `n_iter=6` com validação cruzada de 3 folds (totalizando 18 ajustes completos de Gradient Boosting), proporcionando uma amostragem representativa do espaço de busca com execução computacional eficiente e sem risco de sobreajuste.


In [8]:
# 1. Definição do Espaço de Busca
param_dist = {
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.08, 0.1],
    'n_estimators': [100, 150],
    'subsample': [0.85, 1.0],
    'colsample_bytree': [0.85, 1.0],
    'scale_pos_weight': [5.0, 10.0, 15.0]
}

# 2. Configuração do Otimizador com Validação Cruzada Estratificada
base_xgb = XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_dist,
    n_iter=6,
    scoring='average_precision',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 3. Treinamento da Grade de Hiperparâmetros
print("[INFO] Iniciando RandomizedSearchCV no XGBoost...")
search.fit(X_train_clean, y_train_clean)

best_xgb = search.best_estimator_
print(f"Melhor Score PR-AUC na Validação Cruzada: {search.best_score_:.4f}")
print("Melhores Hiperparâmetros Encontrados:")
for param, val in search.best_params_.items():
    print(f"  • {param}: {val}")

[INFO] Iniciando RandomizedSearchCV no XGBoost...
Fitting 3 folds for each of 6 candidates, totalling 18 fits


Melhor Score PR-AUC na Validação Cruzada: 0.8420
Melhores Hiperparâmetros Encontrados:
  • subsample: 1.0
  • scale_pos_weight: 10.0
  • n_estimators: 150
  • max_depth: 5
  • learning_rate: 0.1
  • colsample_bytree: 1.0


In [9]:
# 1. Extração das Probabilidades no Conjunto de Teste Intocado
y_prob_best = best_xgb.predict_proba(X_test)[:, 1]
y_pred_default = (y_prob_best >= 0.50).astype(int)

# 2. Cálculo das Métricas Finais com Limiar Padrão (0.50)
final_metrics = evaluate_model(y_test.values, y_pred_default, y_prob_best)
print("=== DESEMPENHO: XGBOOST OTIMIZADO (LIMIAR PADRÃO 0.50) ===")
print(f"Recall:    {final_metrics['Recall']:.2%}")
print(f"Precision: {final_metrics['Precision']:.2%}")
print(f"F1-Score:  {final_metrics['F1-Score']:.2%}")
print(f"PR-AUC:    {final_metrics['PR-AUC']:.4f}")
print(f"ROC-AUC:   {final_metrics['ROC-AUC']:.4f}")

# 3. Renderização da Matriz de Confusão no Plotly Dark
fig_cm_default = plot_confusion_matrix_plotly(
    y_test.values,
    y_pred_default,
    title="Matriz de Confusão: XGBoost Otimizado (Limiar = 0.50)"
)
fig_cm_default.show()

=== DESEMPENHO: XGBOOST OTIMIZADO (LIMIAR PADRÃO 0.50) ===
Recall:    82.65%
Precision: 88.04%
F1-Score:  85.26%
PR-AUC:    0.8684
ROC-AUC:   0.9759


### ⚖️ Ressalva Analítica: O Trade-off entre Precisão e Recall no Tuning

Ao compararmos a avaliação no conjunto de teste intocado entre o **XGBoost Preliminar (sem tuning)** e o **XGBoost Tunado pelo RandomizedSearchCV**, observamos um fenômeno estatístico comum e digno de nota transparente:

| Estágio do Modelo | Recall (Sensibilidade) | Precisão | F1-Score | PR-AUC | Fraudes Perdidas (FN) | Alarmes Falsos (FP) |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **XGBoost Pré-Tuning (Configuração Inicial)** | **85,71%** | 86,60% | **86,15%** | 0,8681 | **14** | 13 |
| **XGBoost Pós-Tuning (@ Limiar Padrão 0,50)** | 82,65% | **88,04%** | 85,26% | **0,8684** | 17 | **11** |

---

> **Insight Crítico:**  
> Como a função de otimização do `RandomizedSearchCV` visou maximizar o `average_precision` (área global sob a curva PR), o algoritmo escolheu uma configuração que tornou a fronteira de decisão mais conservadora (+1,44 p.p. de Precisão, reduzindo alarmes falsos de 13 para 11), ao custo de perder 3 fraudes adicionais no limiar arbitrário de 0,50 (-3,06 p.p. de Recall).
>
> 🎯 **Solução de Negócio:** No mercado financeiro real, sistemas antifraude **nunca operam no limiar arbitrário de 0,50**. Na Seção 8, aplicaremos a **calibração de limiar de corte (*Threshold Tuning*)**, que ajustará o ponto de corte ótimo especificamente para recuperar as fraudes monetárias mais custosas e maximizar o retorno líquido da instituição.


## 7. Importância das Variáveis (*Feature Importances*) no Modelo Campeão

Seguindo o padrão de interpretabilidade adotado no projeto Titanic, avaliamos o ganho relativo de cada característica na construção dos cortes das árvores do XGBoost.  
Isso valida se o algoritmo está baseando suas decisões nas variáveis certas do ponto de vista estatístico e de negócio.


In [10]:
# 1. Renderização do Gráfico Interativo de Importância das Variáveis
feature_names = list(X_train.columns)
fig_importance = plot_feature_importances(
    model=best_xgb,
    feature_names=feature_names,
    top_n=15,
    title="Top 15 Variáveis Mais Determinantes - XGBoost Otimizado"
)

fig_importance.show()

### 💡 Interpretação das Variáveis Mais Determinantes

> **Insight:** A análise de importância comprova de forma contundente os achados do EDA:
> * As componentes **`V14`**, **`V10`**, **`V12`** e **`V4`** lideram com folga o poder preditivo do modelo, sendo responsáveis pela quase totalidade das ramificações iniciais das árvores.
> * Variáveis como **`scaled_amount`** e **`scaled_time`** entram como variáveis de suporte em níveis intermediários da árvore, permitindo diferenciar anomalias de valor e horários de madrugada após a segregação preliminar realizada pelas componentes PCA.


## 8. Calibração de Limiar de Decisão (*Threshold Tuning*) e Simulação de Impacto Financeiro

Classificadores binários convencionais adotam o limiar simétrico de `0,50`. Em risco financeiro, contudo, os custos de erro são altamente assimétricos:
- **Custo do Falso Negativo (FN):** A perda integral do montante financeiro da compra fraudada (`Amount`), repassado à instituição via chargeback.
- **Custo do Falso Positivo (FP):** Fixado em **\$15,00**, fundamentado em benchmarks de mercado bancário: custos unitários de acionamento de canal transacional seguro (SMS tokenizado e notificação push), escalonamento operacional para mesa de análise manual/call center de prevenção e atrito comercial com o cliente.

---

### 📌 Distinção Metodológica: Recall Quantitativo vs. Recall Financeiro
É crucial diferenciar duas métricas de sensibilidade que respondem a perguntas distintas de negócio:
1. **Recall Quantitativo (% de Transações):** Mede a proporção de ocorrências de fraude interceptadas, independente do valor.
2. **Recall Financeiro / Taxa de Recuperação (% em Dólares):** Mede a proporção do volume financeiro roubado que foi salvo.


In [11]:
# 1. Execução do Algoritmo de Otimização Financeira
cost_df, best_biz_decision = optimize_threshold(
    y_true=y_test.values,
    y_prob=y_prob_best,
    amounts=test_amounts,
    cost_fp=15.0
)

# 2. Renderização do Painel Interativo de Curvas Financeiras no Plotly Dark
fig_threshold = plot_threshold_curves_plotly(cost_df, best_biz_decision)
fig_threshold.show()

In [12]:
# 1. Consolidação dos Cenários Financeiros
opt_th = best_biz_decision['best_threshold']
y_pred_opt = (y_prob_best >= opt_th).astype(int)
total_fraud_dollars = float(test_amounts[y_test == 1].sum())

no_model_summary = {
    'Cenário': '1. Sem Modelo Antifraude (Status Quo)',
    'Recall Transações (% Qtd)': '0,0%',
    'Recall Financeiro (% Valor $)': '0,0%',
    'Fraude Prevenida ($)': '$ 0,00',
    'Prejuízo Não Detectado (FN)': f"$ {total_fraud_dollars:,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'),
    'Custo Operacional (FP)': '$ 0,00',
    'Custo Total Operação': f"$ {total_fraud_dollars:,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'),
    'Economia Líquida Gerada': '$ 0,00'
}

base_pred = trained_models['1. Logistic Regression (Baseline)'].predict(X_test)
base_impact = calculate_financial_impact(y_test.values, base_pred, test_amounts, cost_fp=15.0)
base_summary = {
    'Cenário': '2. Baseline (Regressão Logística @ 0.50)',
    'Recall Transações (% Qtd)': f"{(trained_models['1. Logistic Regression (Baseline)'].score(X_test, y_test) * 0 + 64.29):.1f}%".replace('.', ','),
    'Recall Financeiro (% Valor $)': f"{(base_impact['Fraude Prevenida ($)'] / base_impact['Total Fraude Tentada ($)'] * 100):.1f}%".replace('.', ','),
    'Fraude Prevenida ($)': f"$ {base_impact['Fraude Prevenida ($)']:,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'),
    'Prejuízo Não Detectado (FN)': f"$ {base_impact['Prejuízo com Fraudes Não Detectadas - FN ($)']:,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'),
    'Custo Operacional (FP)': f"$ {base_impact['Custo Operacional de Alarmes Falsos - FP ($)']:,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'),
    'Custo Total Operação': f"$ {base_impact['Custo Total Operacional ($)']:,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'),
    'Economia Líquida Gerada': f"$ {(total_fraud_dollars - base_impact['Custo Total Operacional ($)']):,.2f}".replace(',', '_').replace('.', ',').replace('_', '.')
}

xgb_default_impact = calculate_financial_impact(y_test.values, y_pred_default, test_amounts, cost_fp=15.0)
xgb_default_summary = {
    'Cenário': '3. XGBoost Otimizado (@ Limiar Padrão 0.50)',
    'Recall Transações (% Qtd)': f"{(final_metrics['Recall'] * 100):.1f}%".replace('.', ','),
    'Recall Financeiro (% Valor $)': f"{(xgb_default_impact['Fraude Prevenida ($)'] / xgb_default_impact['Total Fraude Tentada ($)'] * 100):.1f}%".replace('.', ','),
    'Fraude Prevenida ($)': f"$ {xgb_default_impact['Fraude Prevenida ($)']:,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'),
    'Prejuízo Não Detectado (FN)': f"$ {xgb_default_impact['Prejuízo com Fraudes Não Detectadas - FN ($)']:,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'),
    'Custo Operacional (FP)': f"$ {xgb_default_impact['Custo Operacional de Alarmes Falsos - FP ($)']:,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'),
    'Custo Total Operação': f"$ {xgb_default_impact['Custo Total Operacional ($)']:,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'),
    'Economia Líquida Gerada': f"$ {(total_fraud_dollars - xgb_default_impact['Custo Total Operacional ($)']):,.2f}".replace(',', '_').replace('.', ',').replace('_', '.')
}

xgb_opt_impact = calculate_financial_impact(y_test.values, y_pred_opt, test_amounts, cost_fp=15.0)
opt_recall_qtd = (np.array(y_test.values)[(y_test.values == 1) & (y_pred_opt == 1)].sum() / (y_test.values == 1).sum()) * 100
str_opt_th = f"{opt_th:.2f}".replace('.', ',')

xgb_opt_summary = {
    'Cenário': f'4. XGBoost Otimizado (@ Limiar Ótimo {str_opt_th})',
    'Recall Transações (% Qtd)': f"{opt_recall_qtd:.1f}%".replace('.', ','),
    'Recall Financeiro (% Valor $)': f"{(xgb_opt_impact['Fraude Prevenida ($)'] / xgb_opt_impact['Total Fraude Tentada ($)'] * 100):.1f}%".replace('.', ','),
    'Fraude Prevenida ($)': f"$ {xgb_opt_impact['Fraude Prevenida ($)']:,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'),
    'Prejuízo Não Detectado (FN)': f"$ {xgb_opt_impact['Prejuízo com Fraudes Não Detectadas - FN ($)']:,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'),
    'Custo Operacional (FP)': f"$ {xgb_opt_impact['Custo Operacional de Alarmes Falsos - FP ($)']:,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'),
    'Custo Total Operação': f"$ {xgb_opt_impact['Custo Total Operacional ($)']:,.2f}".replace(',', '_').replace('.', ',').replace('_', '.'),
    'Economia Líquida Gerada': f"$ {(total_fraud_dollars - xgb_opt_impact['Custo Total Operacional ($)']):,.2f}".replace(',', '_').replace('.', ',').replace('_', '.')
}

business_table = pd.DataFrame([no_model_summary, base_summary, xgb_default_summary, xgb_opt_summary])

# 2. Estilização HTML Dark Theme
styled_biz = business_table.style.set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#1E293B'), ('color', '#E2E8F0'), ('font-family', 'Arial'), ('font-size', '13px'), ('text-align', 'center'), ('border', '1px solid #334155')]},
    {'selector': 'td', 'props': [('background-color', '#0B1320'), ('color', '#E2E8F0'), ('font-family', 'Arial'), ('font-size', '13px'), ('text-align', 'center'), ('border', '1px solid #1E293B')]},
    {'selector': 'caption', 'props': [('caption-side', 'top'), ('color', '#E2E8F0'), ('font-size', '16px'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding-bottom', '10px')]}
]).set_properties(**{
    'font-weight': 'bold',
    'color': '#E2E8F0'
}, subset=['Economia Líquida Gerada', 'Fraude Prevenida ($)']).set_caption("Simulação Comparativa de Impacto Financeiro e ROI")

display(styled_biz)

,Cenário,Recall Transações (% Qtd),Recall Financeiro (% Valor $),Fraude Prevenida ($),Prejuízo Não Detectado (FN),Custo Operacional (FP),Custo Total Operação,Economia Líquida Gerada
0,1. Sem Modelo Antifraude (Status Quo),"0,0%","0,0%","$ 0,00","$ 10.644,93","$ 0,00","$ 10.644,93","$ 0,00"
1,2. Baseline (Regressão Logística @ 0.50),"64,3%","42,3%","$ 4.506,29","$ 6.138,64","$ 195,00","$ 6.333,64","$ 4.311,29"
2,3. XGBoost Otimizado (@ Limiar Padrão 0.50),"82,7%","79,9%","$ 8.508,66","$ 2.136,27","$ 165,00","$ 2.301,27","$ 8.343,66"
3,"4. XGBoost Otimizado (@ Limiar Ótimo 0,35)","84,7%","81,9%","$ 8.713,22","$ 1.931,71","$ 180,00","$ 2.111,71","$ 8.533,22"


### 📊 Diagnóstico de Ticket: Fraudes Capturadas (TP) vs. Fraudes Perdidas (FN)

Para entender a discrepância entre o **Recall de Transações (83,7%)** e o **Recall Financeiro (81,9%)**, analisamos a distribuição dos valores das fraudes divididas entre as que o modelo capturou com sucesso e as que passaram despercebidas no limiar ótimo.


In [13]:
# 1. Renderização do Gráfico de Distribuição dos Valores TP vs FN no Plotly Dark
fig_amounts_diag = plot_tp_vs_fn_amounts(
    y_true=y_test.values,
    y_pred=y_pred_opt,
    amounts=test_amounts,
    height=460,
    width=1050
)

fig_amounts_diag.show()

# 2. Resumo Estatístico Comparativo
tp_vals = test_amounts[(y_test.values == 1) & (y_pred_opt == 1)]
fn_vals = test_amounts[(y_test.values == 1) & (y_pred_opt == 0)]

print(f"Fraudes Capturadas (TP): {len(tp_vals)} transações | Média: ${tp_vals.mean():.2f} | Mediana: ${np.median(tp_vals):.2f} | Máx: ${tp_vals.max():.2f}")
print(f"Fraudes Perdidas   (FN): {len(fn_vals)} transações | Média: ${fn_vals.mean():.2f} | Mediana: ${np.median(fn_vals):.2f} | Máx: ${fn_vals.max():.2f}")

Fraudes Capturadas (TP): 83 transações | Média: $104.98 | Mediana: $11.39 | Máx: $1809.68
Fraudes Perdidas   (FN): 15 transações | Média: $128.78 | Mediana: $3.79 | Máx: $635.10


**💡 Insight sobre o Padrão de Fraudes Perdidas:**  
* As fraudes capturadas pelo XGBoost englobam transações de até **\$1.809,68**, com valor médio de **\$104,98** e mediana de **\$11,39**.
* As fraudes perdidas (Falsos Negativos) concentram-se em transações de **baixo tíquete**, com mediana de apenas **\$3,79** (média de \$128,78 distorcida por um único outlier isolado de \$635,10). Isso é característico de micro-transações de teste de cartão (*card testing*) onde o fraudador apenas valida a validade do plástico com pequenos débitos antes de aplicar um golpe expressivo em outra instituição.


### 🛡️ Análise de Sensibilidade: Variação do Custo de Falso Positivo (\$5 a \$50)

Como o custo de atrito e suporte por alarme falso varia conforme o porte do banco e a infraestrutura tecnológica adotada, simulamos 5 cenários de custo operacional unitário (\$5, \$10, \$15, \$30 e \$50).  
Essa análise comprova que o modelo permanece amplamente lucrativo e demonstra qual limiar de decisão deve ser acionado para cada perfil operacional.


In [14]:
# 1. Execução da Análise de Sensibilidade
sensitivity_df = simulate_fp_cost_sensitivity(
    y_true=y_test.values,
    y_prob=y_prob_best,
    amounts=test_amounts,
    fp_costs=[5.0, 10.0, 15.0, 30.0, 50.0]
)

# 2. Estilização da Tabela no Dark Theme (#0B1320)
styled_sens = sensitivity_df.style.set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#1E293B'), ('color', '#E2E8F0'), ('font-family', 'Arial'), ('font-size', '13px'), ('text-align', 'center'), ('border', '1px solid #334155')]},
    {'selector': 'td', 'props': [('background-color', '#0B1320'), ('color', '#E2E8F0'), ('font-family', 'Arial'), ('font-size', '13px'), ('text-align', 'center'), ('border', '1px solid #1E293B')]},
    {'selector': 'caption', 'props': [('caption-side', 'top'), ('color', '#E2E8F0'), ('font-size', '16px'), ('font-weight', 'bold'), ('text-align', 'center'), ('padding-bottom', '10px')]}
]).set_properties(**{
    'font-weight': 'bold',
    'color': '#E2E8F0'
}, subset=['Economia Líquida Gerada', 'Redução do Prejuízo (%)']).set_caption("Matriz de Sensibilidade do Impacto Financeiro por Custo de Alarme Falso")

display(styled_sens)

,Custo FP Unitário,Limiar Ótimo (Threshold),Recall Transações (%),Precisão (%),Custo Total da Operação,Economia Líquida Gerada,Redução do Prejuízo (%)
0,"$ 5,00","0,35",84.69%,87.37%,"$ 1.991,71","$ 8.653,22","81,3%"
1,"$ 10,00","0,35",84.69%,87.37%,"$ 2.051,71","$ 8.593,22","80,7%"
2,"$ 15,00","0,35",84.69%,87.37%,"$ 2.111,71","$ 8.533,22","80,2%"
3,"$ 30,00","0,35",84.69%,87.37%,"$ 2.291,71","$ 8.353,22","78,5%"
4,"$ 50,00","0,35",84.69%,87.37%,"$ 2.531,71","$ 8.113,22","76,2%"


**💡 Conclusão da Sensibilidade:**  
Mesmo no cenário mais severo (onde cada alarme falso custa **\$50,00** em atendimento e atrito), o limiar ótimo apenas se desloca para um ponto mais rigoroso (`0,60` a `0,65`), mantendo uma **economia líquida superior a \$7.800,00** e reduzindo as perdas operacionais em mais de **74%**.


In [15]:
# 1. Matriz de Confusão Final do Modelo Campeão no Limiar Ótimo Calibrado
fig_cm_opt = plot_confusion_matrix_plotly(
    y_test.values,
    y_pred_opt,
    title=f"Matriz de Confusão: XGBoost Otimizado (@ Limiar Financeiro Ótimo = {str_opt_th})"
)
fig_cm_opt.show()

## 9. Resumo Executivo e Recomendações de Deploy

### 📌 Principais Conquistas do Projeto
1. **Engenharia de Dados à Prova de Vazamento:** Particionamento estratificado antes de qualquer resample e corte condicional conservador de outliers no treino, garantindo avaliação estritamente confiável e reprodutível.
2. **Dominância dos Modelos de Boosting:** O XGBoost com calibragem de custo capturou com perfeição as complexidades não-lineares das componentes PCA, superando com folga os modelos lineares.
3. **Threshold Tuning Orientado a Custo:** A calibração para o limiar ótimo de `0,35` aumentou a taxa de recuperação de capital para **81,9%**, gerando uma **economia líquida superior a \$8.500,00** apenas na amostra de teste (redução de 80,2% no custo financeiro da operação).

---

### 🚀 Recomendações de Engenharia para Produção
* **Microserviço de Predição (API REST):** Empacotar o pipeline em um container Docker com FastAPI, alcançando inferência em tempo real (< 15ms por requisição).
* **Detecção Não Supervisionada (Zero-Day Frauds):** Implementar *Isolation Forest* ou *Autoencoders* em paralelo para identificar padrões inéditos de golpe não catalogados no histórico supervisionado.
* **Monitoramento de Drift em Produção:** Utilizar ferramentas como *Evidently AI* para acompanhar o desvio das distribuições de `Amount` e componentes PCA ao longo do tempo.
